In [1]:
import os
import dotenv

dotenv.load_dotenv()

if not os.getenv("GITHUB_TOKEN"):
    raise ValueError("GITHUB_TOKEN is not set")

os.environ["OPENAI_API_KEY"] = os.getenv("GITHUB_TOKEN")
os.environ["OPENAI_BASE_URL"] = "https://models.inference.ai.azure.com/"

In [2]:
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext, load_index_from_storage
from llama_index.core import Settings
import os

llm = OpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_BASE_URL"),
)

embed_model = OpenAIEmbedding(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_BASE_URL"),
)

Settings.embed_model = embed_model

In [3]:
import phoenix as px
px.launch_app()

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/python/3.12.1/lib/python3.12/contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_cumulative_llm_token_count_total
  next(self.gen)
/usr/local/python/3.12.1/lib/python3.12/contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_latency
  next(self.gen)


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix


In [4]:
from openinference.instrumentation.llama_index import LlamaIndexInstrumentor
from phoenix.otel import register

tracer_provider = register()
LlamaIndexInstrumentor().instrument(tracer_provider=tracer_provider)

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: default
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [5]:
# change chunk size and chunk overlap via Settings
Settings.chunk_size = 500
Settings.chunk_overlap = 50

In [6]:
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

try:
    storage_context = StorageContext.from_defaults(
        persist_dir="./local_index_chunked_500_50"
    )

    index = load_index_from_storage(storage_context)

except:
    documents = SimpleDirectoryReader("data").load_data()
    index = VectorStoreIndex.from_documents(documents, insert_batch_size=150)
    index.storage_context.persist(persist_dir="./local_index")


2026-02-03 06:19:43,096 - INFO - HTTP Request: POST https://models.inference.ai.azure.com/embeddings "HTTP/1.1 200 OK"


In [7]:
# solution
query_engine = index.as_query_engine(
  llm=llm
)
query_engine.query("what is vector data base?")

2026-02-03 06:19:43,619 - INFO - HTTP Request: POST https://models.inference.ai.azure.com/embeddings "HTTP/1.1 200 OK"
2026-02-03 06:19:46,616 - INFO - HTTP Request: POST https://models.inference.ai.azure.com/chat/completions "HTTP/1.1 200 OK"


Response(response='A vector database is a type of database designed to store and manage vector embeddings, which are numerical representations of data points in a high-dimensional space. These embeddings are often used in machine learning and AI applications, particularly for tasks like similarity search, where the goal is to find items that are similar to a given query based on their vector representations. Vector databases enable efficient querying and retrieval of these embeddings, facilitating applications such as recommendation systems, image and text search, and more.', source_nodes=[NodeWithScore(node=TextNode(id_='6a85e585-1384-45aa-9cd7-faf8268d3894', embedding=None, metadata={'file_path': '/workspaces/introduction_to_Retrival_Ugmented_generation/data/data.txt', 'file_name': 'data.txt', 'file_type': 'text/plain', 'file_size': 360, 'creation_date': '2026-02-01', 'last_modified_date': '2026-02-01'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_d

In [8]:
query_engine.query("What are the components of rag?")

2026-02-03 06:19:46,945 - INFO - HTTP Request: POST https://models.inference.ai.azure.com/embeddings "HTTP/1.1 200 OK"
2026-02-03 06:19:49,126 - INFO - HTTP Request: POST https://models.inference.ai.azure.com/chat/completions "HTTP/1.1 200 OK"


Response(response='Retrieval-Augmented Generation (RAG) primarily consists of two key components: a retrieval system and a generation model. The retrieval system fetches relevant and up-to-date data from external sources, while the generation model, typically a large language model, uses this information to generate accurate and contextually relevant responses.', source_nodes=[NodeWithScore(node=TextNode(id_='6a85e585-1384-45aa-9cd7-faf8268d3894', embedding=None, metadata={'file_path': '/workspaces/introduction_to_Retrival_Ugmented_generation/data/data.txt', 'file_name': 'data.txt', 'file_type': 'text/plain', 'file_size': 360, 'creation_date': '2026-02-01', 'last_modified_date': '2026-02-01'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOUR